<div style="max-width:760px;padding:22px;border-radius:18px;background:linear-gradient(135deg,#4f46e5,#7c3aed);color:#fff;font-family:sans-serif">
<div style="font-size:13px;font-weight:800;letter-spacing:.08em;opacity:.85">🎙️ CLONAGEM DE VOZ MASTER · v1.5.16</div>
<div style="font-size:26px;font-weight:900;margin:8px 0 14px">Ligue sua voz em 3 passos</div>
<div style="font-size:16px;line-height:2">
<b>1.</b> Clique no botão <b>▶️ (play)</b> do bloco aqui embaixo<br>
<b>2.</b> Se aparecer um <b>aviso do Google</b>, clique em <b>“Executar mesmo assim”</b><br>
<b>3.</b> Espere e clique no <b>botão roxo</b> que vai aparecer 🎉
</div>
<div style="margin-top:14px;font-size:13px;opacity:.85">💡 O aviso aparece porque este caderno foi feito pelo Canivete, não pelo Google. É seguro.</div>
</div>

In [ ]:
#@title 🎙️ CLIQUE NO ▶️ (PLAY) AQUI DO LADO PARA LIGAR SUA VOZ
#@markdown ### Depois é só esperar ☕ — quando aparecer o **botão roxo**, clique nele!
import os, re, subprocess, sys, time, html
from IPython.display import display, HTML

VERSAO = "v1.5.16"
PASTA = "/content/chatterbox-ptbr"
SCRIPT = "/content/iniciar_servidor.py"
REPO = "https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-pt-br"
RAW = "https://raw.githubusercontent.com/jhoncorretor2025-blip/delta-prompts/main/canivete/voz/iniciar_servidor.py"

ETAPAS = ["Ligar a GPU", "Baixar o modelo", "Instalar (demora)", "Ligar a voz", "Pronto!"]

def tela(etapa, titulo, texto="", cor="#8b9cff", extra=""):
    bolinhas = ""
    for i, nome in enumerate(ETAPAS, 1):
        if i < etapa:   fundo, sinal = "#059669", "✓"
        elif i == etapa: fundo, sinal = cor, str(i)
        else:           fundo, sinal = "#2a3550", str(i)
        bolinhas += (f'<div style="flex:1;text-align:center;font:600 12px sans-serif;color:#cbd5e1">'
                     f'<div style="margin:0 auto 6px;width:34px;height:34px;border-radius:50%;background:{fundo};'
                     f'color:#fff;display:grid;place-items:center;font:800 16px sans-serif">{sinal}</div>{nome}</div>')
    return HTML(f'''
<div style="max-width:760px;border-radius:18px;padding:22px;background:linear-gradient(160deg,#151d33,#0e1424);
            border:1px solid #2a3550;color:#f5f7ff;font-family:system-ui,sans-serif">
  <div style="font:800 13px sans-serif;color:#a5b4fc;letter-spacing:.06em">🎙️ CLONAGEM DE VOZ MASTER · {VERSAO}</div>
  <div style="display:flex;gap:6px;margin:16px 0 18px">{bolinhas}</div>
  <div style="font:800 22px sans-serif;margin-bottom:6px">{titulo}</div>
  <div style="font:15px/1.55 sans-serif;color:#cbd5e1">{texto}</div>
  {extra}
</div>''')

painel = display(tela(1, "⏳ Começando…", "Verificando se a GPU está ligada."), display_id=True)

def rodar(etapa, titulo, texto, comando):
    """Roda um comando mostrando o tempo passando. Se der erro, mostra o motivo."""
    inicio = time.time()
    proc = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    saida = []
    import threading
    leitor = threading.Thread(target=lambda: saida.extend(l.rstrip() for l in proc.stdout), daemon=True)
    leitor.start()
    while proc.poll() is None:
        seg = int(time.time() - inicio)
        painel.update(tela(etapa, titulo, f"{texto}<br><br>⏱️ <b>{seg//60}:{seg%60:02d}</b> — pode deixar rodando, não precisa clicar em nada."))
        time.sleep(1)
    leitor.join(5)
    if proc.returncode != 0:
        log = html.escape("\n".join(saida[-15:]))
        painel.update(tela(etapa, "❌ Deu um problema", "Tire um <b>print desta tela</b> e mande para o Claude no chat. 🙏", "#e11d48",
            f'<pre style="margin-top:14px;padding:12px;border-radius:10px;background:#0b1020;color:#fda4af;white-space:pre-wrap;font-size:12px">{log}</pre>'))
        raise SystemExit
    return saida

# 1) GPU
if subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode != 0:
    painel.update(tela(1, "⚠️ A GPU está desligada", "Faça só isto e depois clique no ▶️ de novo:", "#f59e0b",
        '<ol style="font:15px/1.9 sans-serif;color:#fde68a;margin-top:10px">'
        '<li>No menu de cima, clique em <b>Ambiente de execução</b></li>'
        '<li>Clique em <b>Alterar tipo de ambiente de execução</b></li>'
        '<li>Escolha <b>T4 GPU</b> (ou <b>L4</b>) e clique em <b>Salvar</b></li>'
        '<li>Clique no <b>▶️</b> deste bloco de novo</li></ol>'))
    raise SystemExit

# 2) Modelo
if not os.path.exists(PASTA + "/app.py"):
    rodar(2, "📥 Baixando o modelo de voz…", "Pegando o Chatterbox em português do Brasil.",
          f"git clone -q --depth 1 {REPO} {PASTA}")

# 3) Instalação
if not os.path.exists("/content/.voz_instalado"):
    rodar(3, "📦 Instalando as peças…", "Essa é a parte mais demorada (uns <b>5 a 10 minutos</b> na primeira vez). ☕",
          f"pip install -q -r {PASTA}/requirements.txt && touch /content/.voz_instalado")
rodar(3, "📦 Quase lá…", "Atualizando o programa do servidor.", f"wget -q -O {SCRIPT} {RAW}")

# 4) Ligar
painel.update(tela(4, "🧠 Ligando a voz…", "Carregando o modelo na GPU. Leva de <b>1 a 3 minutos</b>."))
proc = subprocess.Popen([sys.executable, "-u", SCRIPT, "--motor", "colab", "--pasta", PASTA],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/content")
inicio, ultimas, link = time.time(), [], None
for linha in proc.stdout:
    ultimas = (ultimas + [linha.rstrip()])[-15:]
    achou = re.search(r"(https://jhoncorretor2025-blip\.github\.io/\S+servidor=\S+)", linha)
    if achou and not link:
        link = achou.group(1)
        painel.update(tela(6, "✅ SUA VOZ ESTÁ LIGADA!", "Clique no botão abaixo. A página vai abrir <b>já conectada</b> ao Colab.", "#059669",
            f'<a href="{link}" target="_blank" style="display:block;margin:18px 0 12px;padding:18px;border-radius:14px;'
            f'background:linear-gradient(135deg,#8b9cff,#a855f7);color:#fff;text-align:center;font:800 20px sans-serif;'
            f'text-decoration:none">🎙️ ABRIR MINHA PÁGINA DE VOZ</a>'
            '<div style="font:14px/1.6 sans-serif;color:#fde68a">⚠️ <b>Deixe esta aba do Colab aberta</b> enquanto usa.<br>'
            'O ▶️ vai continuar girando — <b>é normal</b>, significa que a voz está ligada.<br>'
            'Para desligar, clique no ⏹️ deste bloco.</div>'))
    elif not link:
        seg = int(time.time() - inicio)
        painel.update(tela(4, "🧠 Ligando a voz…", f"Carregando o modelo na GPU. Leva de <b>1 a 3 minutos</b>.<br><br>⏱️ <b>{seg//60}:{seg%60:02d}</b>"))
proc.wait()
log = html.escape("\n".join(ultimas))
painel.update(tela(4, "⏹️ A voz foi desligada", "Se você não desligou de propósito, tire um <b>print</b> e mande para o Claude. Para ligar de novo, clique no ▶️.", "#e11d48",
    f'<pre style="margin-top:14px;padding:12px;border-radius:10px;background:#0b1020;color:#fda4af;white-space:pre-wrap;font-size:12px">{log}</pre>'))
